# Kaggle SPICE Self-Play Training

This notebook implements the **SPICE (Self-Play In Corpus Environments)** framework for OnCallEnv Red Shift. It trains a single model to act as both an **LLM Attacker** (generating hard scenarios) and an **LLM Defender** (solving them) using **DrGRPO**.

## 1. GPU Check

Expected: Two Tesla T4 GPUs.

In [17]:
!nvidia-smi

Sat Apr 25 11:19:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Bootstrap Repository

Clones or updates the repository in `/kaggle/working`.

In [18]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "spicy-attacker"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            spicy-attacker -> FETCH_HEAD
   cddd9c0..4562fa7  spicy-attacker -> origin/spicy-attacker
Already on 'spicy-attacker'


Your branch is behind 'origin/spicy-attacker' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating cddd9c0..4562fa7
Fast-forward
 notebooks/05_kaggle_spice_selfplay.ipynb | 288 +++++++++++++++++++++++++------
 scripts/spice_defender.py                |  29 ++--
 scripts/train_spice_selfplay.py          |   4 +-
 3 files changed, 260 insertions(+), 61 deletions(-)
cwd: /kaggle/working/MetaHackathon-R2
4562fa7 changed reward to max!
be6435c sim-error?
cddd9c0 error fixed
f5558b1 more output decoding
05e30c0 defender output decoding


From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            spicy-attacker -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [ ]:
!python scripts/debug_spice_iter.py


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 15:11:30.887750: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777129890.910036    2586 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777129890.917767    2586 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777129890.938010    2586 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777129890.938041    2586 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

## 3. Install Dependencies

Installs standard requirements and LLM training stack (Unsloth, TRL).

In [3]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt
!python -m pip install pytest

## 4. Global Environment Setup

In [13]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
import sys
sys.path.append("src")
sys.path.append("scripts")

/kaggle/working
env: PYTHONPATH=src:scripts


## 5. Verify Attacker Logic

Run the unit tests to ensure the discrete action parser and variance rewards are correct on this kernel.

In [14]:
!python -m pytest tests/test_llm_attacker.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/MetaHackathon-R2
configfile: pyproject.toml
plugins: anyio-4.12.1, langsmith-0.7.6, typeguard-4.5.1
collected 25 items                                                             

tests/test_llm_attacker.py::TestParseAttackerActions::test_single_valid_action PASSED [  4%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_multiple_valid_actions PASSED [  8%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_invalid_field_name_ignored PASSED [ 12%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_invalid_value_ignored PASSED [ 16%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_no_actions_returns_invalid PASSED [ 20%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_empty_actions_block PASSED [ 24%]
tests/test_llm_attacker.py::Tes

## 6. SPICE Self-Play Smoke Run

Tests the full interaction loop between Attacker and Defender with minimal steps.

In [ ]:
!python scripts/train_spice_selfplay.py \
    --selfplay-iterations 2 \
    --group-size 2 \
    --batch-size 2 \
    --max-steps 5 \
    --out-dir "training_results/spice_smoke" \
    --report-to "none" \
    --verbose

Loaded 120 parent scenarios
/kaggle/working/MetaHackathon-R2/scripts/train_spice_selfplay.py:578: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 16:26:58.790618: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777134418.814164    3159 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777134418.823130    3159 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register facto

## 7. Main SPICE Self-Play Training

Runs the primary co-evolution loop. Adjust `--report-to` to `wandb` if you have it configured.

In [ ]:
!pip uninstall -y vllm


Found existing installation: vllm 0.19.1
Uninstalling vllm-0.19.1:
  Successfully uninstalled vllm-0.19.1


In [19]:
!python scripts/train_spice_selfplay.py \
    --model-name "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" \
    --selfplay-iterations 5 \
    --group-size 4 \
    --selfplay-group-size 4 \
    --batch-size 2 \
    --max-steps 300 \
    --per-device-train-batch-size 1 \
    --gradient-accumulation-steps 8 \
    --lr 5e-6 \
    --report-to "tensorboard" \
    --out-dir "training_results/spice_selfplay_v2" \
    --verbose

Loaded 120 parent scenarios
/kaggle/working/MetaHackathon-R2/scripts/train_spice_selfplay.py:578: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 16:43:05.889076: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777135385.923783    3319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777135385.933437    3319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register facto

## 8. Summary & Results

In [ ]:
!cat training_results/spice_selfplay_main/summary.json

In [ ]:
import shutil
# Zip the entire results folder
shutil.make_archive('spice_training_results', 'zip', 'training_results/spice_selfplay_v2')
print("Zipping complete: spice_training_results.zip")


In [ ]:
from IPython.display import FileLink
FileLink('spice_training_results.zip')


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def plot_spice_results(run_dir_path):
    run_dir = Path(run_dir_path)
    
    # --- Part 1: Iteration-wise Co-evolution (from Generations) ---
    # Since summary.json is only saved at the end, we look for generation files
    a_gen_path = run_dir / "attacker_generations.json"
    d_gen_path = run_dir / "defender_generations.json"
    
    if a_gen_path.exists() and d_gen_path.exists():
        with open(a_gen_path, "r") as f: a_data = json.load(f)
        with open(d_gen_path, "r") as f: d_data = json.load(f)
        
        # Plotting Attacker vs Defender co-evolution
        # Note: If iterations aren't explicitly labeled, we can still see the trend
        a_rewards = [row.get("reward", 0) for row in a_data]
        d_rewards = [row.get("reward", row.get("raw_reward", 0)) for row in d_data]
        
        plt.figure(figsize=(10, 4))
        plt.plot(a_rewards, label="Attacker Reward", alpha=0.5, color="red")
        plt.plot(d_rewards, label="Defender Reward", alpha=0.5, color="blue")
        plt.title("Self-Play Generation Rewards (Raw Samples)")
        plt.xlabel("Sample Index")
        plt.ylabel("Reward")
        plt.legend()
        plt.grid(True, alpha=0.2)
        plt.show()

    # --- Part 2: GRPO Training (from trainer_state.json) ---
    state_path = run_dir / "trainer_state.json"
    if not state_path.exists():
        checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda x: int(x.name.split("-")[1]))
        if checkpoints:
            state_path = checkpoints[-1] / "trainer_state.json"
            
    if state_path.exists():
        with open(state_path, "r") as f:
            state = json.load(f)
        
        history = state.get("log_history", [])
        metrics = []
        for row in history:
            if "loss" in row or "reward" in row:
                metrics.append({
                    "step": row.get("step"),
                    "loss": row.get("loss"),
                    "reward": row.get("reward"),
                    "kl": row.get("kl")
                })
        
        if metrics:
            df = pd.DataFrame(metrics).dropna(subset=["step"])
            df.to_csv(run_dir / "spice_training_metrics.csv", index=False)
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
            
            # GRPO Reward
            ax1.plot(df["step"], df["reward"], color="green", label="GRPO Aggregate Reward")
            ax1.set_title("GRPO Training Reward")
            ax1.set_xlabel("Step")
            ax1.grid(True, alpha=0.3)
            
            # GRPO Loss
            ax2.plot(df["step"], df["loss"], color="orange", label="GRPO Loss")
            ax2.set_title("GRPO Training Loss")
            ax2.set_xlabel("Step")
            ax2.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            print(f"GRPO metrics stored in {run_dir / 'spice_training_metrics.csv'}")
    else:
        print("Training state not found yet. GRPO phase might not have started.")

# Run for your SPICE directory
plot_spice_results("training_results/spice_selfplay_v2")
